# EDA - MrScraper Price Intelligence

This notebook explores the train/test data before modeling. The goal is to understand:

- train/test timing and overlap
- missingness patterns
- price distribution and outliers
- entity coverage between train and test
- product/shop/category historical stability
- anchor availability in the test file
- where validation errors are likely to come from

The modeling pipeline lives in `src/` and the main report is in `notebook.ipynb`.


In [ ]:
from pathlib import Path
import os
import sys

os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sys.path.append(str(Path.cwd()))

from src.config import PipelineConfig
from src.data import load_data, basic_preprocess

pd.set_option("display.max_columns", 120)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

config = PipelineConfig()
train_raw, test_raw = load_data(config.train_path, config.test_path)
train = basic_preprocess(train_raw, config)
test = basic_preprocess(test_raw, config)

print("Train:", train.shape)
print("Test:", test.shape)

FINAL_VALIDATION_LABEL = "anchor_gated_fallback_calibration_entity_blend_no_calibration"
REPORT_VARIANTS = [
    FINAL_VALIDATION_LABEL,
    "last_price_baseline",
    "entity_blend_calibrated",
    "entity_blend_no_calibration",
    "second_stage_residual",
    "global_calibrated",
    "global_no_calibration",
]


## 1. Dataset Overview

Basic shape, date range, and missing target count. This confirms the outage setup and whether train/test are split by date or timestamp.


In [ ]:
def dataset_overview(df, name):
    return {
        "dataset": name,
        "rows": len(df),
        "columns": df.shape[1],
        "min_capturedAt": df[config.date_col].min(),
        "max_capturedAt": df[config.date_col].max(),
        "missing_price": df[config.target].isna().sum(),
        "known_price": df[config.target].notna().sum(),
        "unique_shopId": df["shopId"].nunique(),
        "unique_itemId": df["itemId"].nunique(),
        "unique_modelId": df["modelId"].nunique(),
    }

pd.DataFrame([
    dataset_overview(train, "train"),
    dataset_overview(test, "test"),
])


In [ ]:
print("Train max timestamp:", train[config.date_col].max())
print("Test min timestamp: ", test[config.date_col].min())
print("Gap:", test[config.date_col].min() - train[config.date_col].max())


## 2. Schema And Missingness

Missingness matters because the test file has many blank feature columns. This table shows which fields are useful at prediction time and which are mostly unavailable.


In [ ]:
def missingness_table(train_df, test_df):
    rows = []
    for col in train_df.columns:
        rows.append({
            "column": col,
            "train_dtype": str(train_df[col].dtype),
            "test_dtype": str(test_df[col].dtype) if col in test_df.columns else "missing_column",
            "train_missing_pct": train_df[col].isna().mean() * 100,
            "test_missing_pct": test_df[col].isna().mean() * 100 if col in test_df.columns else np.nan,
            "train_nunique": train_df[col].nunique(dropna=True),
            "test_nunique": test_df[col].nunique(dropna=True) if col in test_df.columns else np.nan,
        })
    return pd.DataFrame(rows).sort_values("test_missing_pct", ascending=False)

missingness = missingness_table(train_raw, test_raw)
missingness


In [ ]:
missingness.plot(
    x="column",
    y=["train_missing_pct", "test_missing_pct"],
    kind="bar",
    figsize=(14, 5),
    title="Missingness by Column",
)
plt.xticks(rotation=80, ha="right")
plt.ylabel("Missing %")
plt.tight_layout()
plt.show()


## 3. Rows Over Time

Counts by day show scrape density, validation-day size, and the outage window.


In [ ]:
train_daily = train.groupby("date").size().rename("train_rows")
test_daily = test.groupby("date").size().rename("test_rows")
daily_counts = pd.concat([train_daily, test_daily], axis=1).fillna(0).astype(int)
daily_counts.tail(20)


In [ ]:
daily_counts.plot(kind="bar", figsize=(16, 5), title="Rows per Date: Train vs Test")
plt.ylabel("Rows")
plt.tight_layout()
plt.show()


## 4. Price Distribution

Prices are heavily skewed, so the pipeline models `log1p(price)`. These plots show why raw-price errors and percentage errors can behave differently.


In [ ]:
price = train[config.target].dropna().astype(float)
price_summary = price.describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99])
price_summary = price_summary.to_frame("price")
price_summary

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

price.clip(upper=price.quantile(0.99)).hist(bins=80, ax=axes[0])
axes[0].set_title("Price Distribution, Clipped at P99")
axes[0].set_xlabel("price")

np.log1p(price).hist(bins=80, ax=axes[1])
axes[1].set_title("log1p(price) Distribution")
axes[1].set_xlabel("log1p(price)")

plt.tight_layout()
plt.show()


In [ ]:
train.assign(price_bucket=pd.qcut(train[config.target], q=10, duplicates="drop"))     .groupby("price_bucket", observed=True)     .agg(
        rows=(config.target, "size"),
        min_price=(config.target, "min"),
        median_price=(config.target, "median"),
        max_price=(config.target, "max"),
        unique_modelId=("modelId", "nunique"),
    )


## 5. Entity Coverage Between Train And Test

The PDF states that the remaining test products have appeared in training before. This section checks that claim at multiple granularities because "product" can mean `itemId`, `modelId`, or the full `shopId`/`itemId`/`modelId` listing key.


In [ ]:
def coverage(train_df, test_df, col):
    train_set = set(train_df[col].dropna().astype(str))
    test_values = test_df[col].dropna().astype(str)
    seen_mask = test_values.isin(train_set)
    return {
        "entity": col,
        "test_unique": test_values.nunique(),
        "test_rows": len(test_values),
        "rows_seen_in_train": seen_mask.sum(),
        "rows_unseen_in_train": (~seen_mask).sum(),
        "row_seen_pct": seen_mask.mean() * 100 if len(test_values) else np.nan,
        "unique_seen_pct": len(set(test_values) & train_set) / max(test_values.nunique(), 1) * 100,
    }

coverage_df = pd.DataFrame([coverage(train, test, col) for col in ["shopId", "itemId", "modelId", "cat_id", "brand"]])
coverage_df


In [ ]:
coverage_df.plot(
    x="entity",
    y=["row_seen_pct", "unique_seen_pct"],
    kind="bar",
    figsize=(8, 4),
    title="Test Entity Coverage in Train",
)
plt.ylabel("Coverage %")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


### Assignment Claim Check: Did Test Products Appear In Train?

This stricter check reports row-level and unique-key coverage for all test rows, hidden rows only, and anchor rows only. The most important rows for the assignment are `itemId` and the composite listing keys.


In [ ]:
def make_key_frame(df, cols):
    """Return a normalized key frame for robust train/test membership checks."""
    return df[list(cols)].astype("string").fillna("<NA>")


def key_tuples(df, cols):
    key_frame = make_key_frame(df, cols)
    if len(cols) == 1:
        return pd.Series(key_frame[cols[0]].to_numpy(), index=df.index, name="|".join(cols))
    return pd.Series(list(key_frame.itertuples(index=False, name=None)), index=df.index, name="|".join(cols))


def train_membership_summary(train_df, candidate_df, cols, segment_name):
    train_keys = set(key_tuples(train_df, cols).dropna())
    candidate_keys = key_tuples(candidate_df, cols).dropna()
    seen_mask = candidate_keys.isin(train_keys)
    unique_candidate_keys = set(candidate_keys)
    unique_seen_keys = unique_candidate_keys & train_keys
    unique_unseen_keys = unique_candidate_keys - train_keys

    return {
        "segment": segment_name,
        "key": " + ".join(cols),
        "test_rows_checked": len(candidate_keys),
        "rows_seen_in_train": int(seen_mask.sum()),
        "rows_unseen_in_train": int((~seen_mask).sum()),
        "row_seen_pct": seen_mask.mean() * 100 if len(candidate_keys) else np.nan,
        "test_unique_keys": len(unique_candidate_keys),
        "unique_seen_in_train": len(unique_seen_keys),
        "unique_unseen_in_train": len(unique_unseen_keys),
        "unique_seen_pct": len(unique_seen_keys) / len(unique_candidate_keys) * 100 if unique_candidate_keys else np.nan,
    }


coverage_key_specs = [
    ("shop", ["shopId"]),
    ("item/product", ["itemId"]),
    ("model/variant", ["modelId"]),
    ("shop_item", ["shopId", "itemId"]),
    ("item_model", ["itemId", "modelId"]),
    ("full_listing", ["shopId", "itemId", "modelId"]),
]

test_hidden = test[test[config.target].isna()].copy()
test_anchor_rows = test[test[config.target].notna()].copy()
segments_to_check = {
    "all_test_rows": test,
    "hidden_missing_price_rows": test_hidden,
    "known_anchor_rows": test_anchor_rows,
}

claim_rows = []
for segment_name, segment_df in segments_to_check.items():
    for _, cols in coverage_key_specs:
        claim_rows.append(train_membership_summary(train, segment_df, cols, segment_name))

claim_coverage = pd.DataFrame(claim_rows)
claim_coverage


In [ ]:
claim_pivot = (
    claim_coverage
    .pivot(index="key", columns="segment", values=["row_seen_pct", "unique_seen_pct", "rows_unseen_in_train", "unique_unseen_in_train"])
    .sort_index()
)
claim_pivot


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

plot_rows = claim_coverage.query("segment == 'hidden_missing_price_rows'").copy()
y_min = max(0, min(plot_rows["row_seen_pct"].min(), plot_rows["unique_seen_pct"].min()) - 0.1)
y_max = 100.01

plot_rows.plot(x="key", y="row_seen_pct", kind="bar", ax=axes[0], legend=False, color="#4C78A8")
axes[0].set_title("Hidden Test Row Coverage in Train")
axes[0].set_ylabel("row coverage %")
axes[0].set_ylim(y_min, y_max)
axes[0].tick_params(axis="x", rotation=45)

plot_rows.plot(x="key", y="unique_seen_pct", kind="bar", ax=axes[1], legend=False, color="#F58518")
axes[1].set_title("Hidden Test Unique-Key Coverage in Train")
axes[1].set_ylabel("unique coverage %")
axes[1].set_ylim(y_min, y_max)
axes[1].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()


### Inspect Any Unseen Test Keys

If the strict `modelId` or composite-key checks are below 100%, inspect the actual rows. This helps determine whether the PDF claim holds at `itemId` level but not at variant/listing level.


In [ ]:
def unseen_key_rows(train_df, test_df, cols):
    train_keys = set(key_tuples(train_df, cols).dropna())
    test_keys = key_tuples(test_df, cols).dropna()
    unseen_mask = ~test_keys.isin(train_keys)
    result = test_df.loc[unseen_mask, [config.date_col, "date", "shopId", "itemId", "modelId", config.target]].copy()
    result["checked_key"] = " + ".join(cols)
    result["key_value"] = test_keys.loc[unseen_mask].astype(str).values
    return result


unseen_checks = []
for _, cols in coverage_key_specs:
    unseen = unseen_key_rows(train, test, cols)
    if len(unseen):
        unseen_checks.append(unseen)

if unseen_checks:
    unseen_test_keys = pd.concat(unseen_checks, ignore_index=True)
else:
    unseen_test_keys = pd.DataFrame(columns=[config.date_col, "date", "shopId", "itemId", "modelId", config.target, "checked_key", "key_value"])

display(unseen_test_keys)

coverage_output_dir = config.output_dir / "eda_coverage"
coverage_output_dir.mkdir(parents=True, exist_ok=True)
claim_coverage.to_csv(coverage_output_dir / "train_test_entity_coverage.csv", index=False)
unseen_test_keys.to_csv(coverage_output_dir / "unseen_test_keys.csv", index=False)
print(f"Saved coverage outputs to {coverage_output_dir}")


### Interpretation

- If `itemId` is 100%, the assignment claim is true at the product-listing level.
- If `modelId` or `itemId + modelId` is below 100%, some variants are new even though their parent item appeared in training.
- If `full_listing` is below 100%, some exact shop/item/model combinations are new and should use item-level or broader fallback features.


### Product Frequency In Prior Training History

Coverage only tells us whether a product appeared at least once. This section measures how often each test product appeared in training before the test timestamp. It uses `itemId` as the main product definition and also reports variant/listing-level frequency for `modelId`, `shopId + itemId`, `itemId + modelId`, and `shopId + itemId + modelId`.


In [ ]:
def historical_frequency_features(train_df, test_df, cols, key_name):
    train_tmp = train_df.copy()
    test_tmp = test_df.copy()
    train_tmp["_freq_key"] = key_tuples(train_tmp, cols)
    test_tmp["_freq_key"] = key_tuples(test_tmp, cols)

    train_summary = (
        train_tmp
        .groupby("_freq_key", dropna=False)
        .agg(
            train_seen_count=(config.target, "size"),
            train_seen_days=("date", "nunique"),
            first_seen_at=(config.date_col, "min"),
            last_seen_at=(config.date_col, "max"),
            median_train_price=(config.target, "median"),
        )
        .reset_index()
    )

    freq = test_tmp[[config.date_col, "date", "shopId", "itemId", "modelId", config.target, "_freq_key"]].merge(
        train_summary, on="_freq_key", how="left"
    )
    freq["key"] = key_name
    freq["key_value"] = freq["_freq_key"].astype(str)
    freq["is_anchor"] = freq[config.target].notna()
    freq["train_seen_count"] = freq["train_seen_count"].fillna(0).astype(int)
    freq["train_seen_days"] = freq["train_seen_days"].fillna(0).astype(int)
    freq["hours_since_last_seen"] = (
        (freq[config.date_col] - freq["last_seen_at"]).dt.total_seconds() / 3600
    )
    freq["train_frequency_bucket"] = pd.cut(
        freq["train_seen_count"],
        bins=[-0.1, 0, 1, 3, 7, 14, 30, 100, np.inf],
        labels=["0", "1", "2-3", "4-7", "8-14", "15-30", "31-100", "100+"],
    )
    return freq.drop(columns=["_freq_key"])


frequency_specs = [
    ("itemId", ["itemId"]),
    ("modelId", ["modelId"]),
    ("shopId + itemId", ["shopId", "itemId"]),
    ("itemId + modelId", ["itemId", "modelId"]),
    ("shopId + itemId + modelId", ["shopId", "itemId", "modelId"]),
]

frequency_frames = [
    historical_frequency_features(train, test, cols, key_name)
    for key_name, cols in frequency_specs
]
test_history_frequency = pd.concat(frequency_frames, ignore_index=True)
test_history_frequency.head()


In [ ]:
frequency_summary = (
    test_history_frequency
    .groupby(["key", "is_anchor"], dropna=False)
    .agg(
        test_rows=("train_seen_count", "size"),
        unique_test_keys=("key_value", "nunique"),
        unseen_rows=("train_seen_count", lambda s: int((s == 0).sum())),
        median_train_seen_count=("train_seen_count", "median"),
        p10_train_seen_count=("train_seen_count", lambda s: s.quantile(0.10)),
        p25_train_seen_count=("train_seen_count", lambda s: s.quantile(0.25)),
        p75_train_seen_count=("train_seen_count", lambda s: s.quantile(0.75)),
        p90_train_seen_count=("train_seen_count", lambda s: s.quantile(0.90)),
        median_train_seen_days=("train_seen_days", "median"),
        median_hours_since_last_seen=("hours_since_last_seen", "median"),
    )
    .reset_index()
)
frequency_summary


In [ ]:
item_frequency = test_history_frequency.query("key == 'itemId'").copy()
model_frequency = test_history_frequency.query("key == 'modelId'").copy()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

item_frequency["train_frequency_bucket"].value_counts().reindex(["0", "1", "2-3", "4-7", "8-14", "15-30", "31-100", "100+"]).plot(
    kind="bar", ax=axes[0], color="#4C78A8"
)
axes[0].set_title("Test Rows by Prior itemId Frequency")
axes[0].set_xlabel("train rows before test")
axes[0].set_ylabel("test rows")
axes[0].tick_params(axis="x", rotation=0)

model_frequency["train_frequency_bucket"].value_counts().reindex(["0", "1", "2-3", "4-7", "8-14", "15-30", "31-100", "100+"]).plot(
    kind="bar", ax=axes[1], color="#F58518"
)
axes[1].set_title("Test Rows by Prior modelId Frequency")
axes[1].set_xlabel("train rows before test")
axes[1].set_ylabel("test rows")
axes[1].tick_params(axis="x", rotation=0)

plt.tight_layout()
plt.show()


In [ ]:
product_frequency = (
    item_frequency
    .groupby(["itemId", "key_value"], dropna=False)
    .agg(
        test_rows=("itemId", "size"),
        has_anchor=("is_anchor", "any"),
        train_seen_count=("train_seen_count", "first"),
        train_seen_days=("train_seen_days", "first"),
        first_seen_at=("first_seen_at", "first"),
        last_seen_at=("last_seen_at", "first"),
        min_hours_since_last_seen=("hours_since_last_seen", "min"),
        median_train_price=("median_train_price", "first"),
    )
    .reset_index(drop=False)
    .sort_values(["train_seen_count", "test_rows"], ascending=[True, False])
)

display(product_frequency.head(25))
display(product_frequency[["train_seen_count", "train_seen_days", "test_rows", "min_hours_since_last_seen"]].describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.99]))


In [ ]:
low_frequency_products = product_frequency.query("train_seen_count <= 3").copy()
low_frequency_test_rows = item_frequency[item_frequency["itemId"].isin(low_frequency_products["itemId"])].copy()

print("Low-frequency itemIds with <= 3 train rows:", len(low_frequency_products))
print("Test rows belonging to those itemIds:", len(low_frequency_test_rows))
display(low_frequency_products.head(50))

coverage_output_dir = config.output_dir / "eda_coverage"
coverage_output_dir.mkdir(parents=True, exist_ok=True)
test_history_frequency.to_csv(coverage_output_dir / "test_product_history_frequency_by_row.csv", index=False)
frequency_summary.to_csv(coverage_output_dir / "test_product_history_frequency_summary.csv", index=False)
product_frequency.to_csv(coverage_output_dir / "test_item_history_frequency.csv", index=False)
low_frequency_products.to_csv(coverage_output_dir / "low_frequency_test_items.csv", index=False)
print(f"Saved frequency outputs to {coverage_output_dir}")


### Frequency Interpretation

- High `train_seen_count` and low `hours_since_last_seen` mean recent entity priors should be reliable.
- Low-frequency `itemId` rows are the rows where the global model, broader shop/category priors, or anchor-gated fallback matter most.
- If `itemId` has high frequency but `modelId` has zero frequency, the product was seen before but the specific variant is new. In that case, item-level fallback is more defensible than model-level fallback.


## 6. Historical Stability By Entity

The entity blend assumes products/models have stable historical prices. This section measures historical count and log-price variability by key.


In [ ]:
def entity_stability(df, key):
    tmp = df.dropna(subset=[config.target]).copy()
    tmp["target_log"] = np.log1p(tmp[config.target].clip(lower=0))
    agg = tmp.groupby(key)["target_log"].agg(["count", "median", "std", "min", "max"]).reset_index()
    agg["range_log"] = agg["max"] - agg["min"]
    return agg

stability_summaries = []
for key in ["modelId", "itemId", "shopId", "cat_id", "brand"]:
    agg = entity_stability(train, key)
    stability_summaries.append({
        "key": key,
        "entities": len(agg),
        "median_count": agg["count"].median(),
        "p90_count": agg["count"].quantile(0.90),
        "median_std_log": agg["std"].median(),
        "p90_std_log": agg["std"].quantile(0.90),
        "median_range_log": agg["range_log"].median(),
        "p90_range_log": agg["range_log"].quantile(0.90),
    })

pd.DataFrame(stability_summaries)


In [ ]:
model_stability = entity_stability(train, "modelId")
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

model_stability["count"].clip(upper=model_stability["count"].quantile(0.99)).hist(bins=60, ax=axes[0])
axes[0].set_title("modelId Historical Count, Clipped at P99")
axes[0].set_xlabel("rows per modelId")

model_stability["std"].dropna().clip(upper=model_stability["std"].quantile(0.99)).hist(bins=60, ax=axes[1])
axes[1].set_title("modelId Historical log(price) Std, Clipped at P99")
axes[1].set_xlabel("std of log1p(price)")

plt.tight_layout()
plt.show()


## 7. Recent Price Movement

This checks whether last observed prices might be useful. Large recent movement means simple long-term medians may miss current prices.


In [ ]:
train_sorted = train.sort_values(["modelId", config.date_col]).copy()
train_sorted["prev_price"] = train_sorted.groupby("modelId")[config.target].shift(1)
train_sorted["prev_log_price"] = np.log1p(train_sorted["prev_price"].clip(lower=0))
train_sorted["log_price"] = np.log1p(train_sorted[config.target].clip(lower=0))
train_sorted["abs_log_change"] = (train_sorted["log_price"] - train_sorted["prev_log_price"]).abs()

movement = train_sorted.dropna(subset=["abs_log_change"])["abs_log_change"]
movement.describe(percentiles=[0.5, 0.75, 0.9, 0.95, 0.99]).to_frame("abs_log_change")


In [ ]:
movement.clip(upper=movement.quantile(0.99)).hist(bins=80, figsize=(10, 4))
plt.title("Absolute Change From Previous modelId Observation, log space")
plt.xlabel("abs log change")
plt.tight_layout()
plt.show()


## 8. Test Anchor Rows

Known prices in the test file are anchors. This checks how many anchors exist per day and what parts of the marketplace they cover.


In [ ]:
test_anchors = test[test[config.target].notna()].copy()
anchor_summary = test_anchors.groupby("date").agg(
    anchors=(config.target, "size"),
    unique_shopId=("shopId", "nunique"),
    unique_itemId=("itemId", "nunique"),
    unique_modelId=("modelId", "nunique"),
    median_price=(config.target, "median"),
    min_price=(config.target, "min"),
    max_price=(config.target, "max"),
)
anchor_summary


In [ ]:
if len(test_anchors):
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    np.log1p(test_anchors[config.target]).hist(bins=30, ax=axes[0])
    axes[0].set_title("Test Anchor log1p(price)")
    axes[0].set_xlabel("log1p(price)")

    test_anchors.groupby("date").size().plot(kind="bar", ax=axes[1], title="Anchors per Test Date")
    axes[1].set_ylabel("anchors")
    plt.tight_layout()
    plt.show()
else:
    print("No known test anchor prices found.")


In [ ]:
if len(test_anchors):
    pd.DataFrame([coverage(train, test_anchors, col) for col in ["shopId", "itemId", "modelId", "cat_id", "brand"]])
else:
    print("No known test anchor prices found.")


## 9. Validation Error Context

This section turns the saved validation outputs into compact diagnostic tables. The goal is to show which strategy wins, where errors concentrate, and whether calibration helps the weaker fallback models.


In [ ]:
validation_summary_path = config.output_dir / "validation_summary.csv"
validation_segments_path = config.output_dir / "validation_segments.csv"

if validation_summary_path.exists():
    validation_summary = pd.read_csv(validation_summary_path).sort_values("MAE").reset_index(drop=True)
    display(validation_summary)

    fig, ax = plt.subplots(figsize=(10, 4))
    plot_summary = validation_summary.head(8).copy()
    plot_summary.plot(x="base_model", y="MAE", kind="bar", legend=False, ax=ax, color="#4C78A8")
    ax.set_title("Validation MAE by Strategy")
    ax.set_xlabel("")
    ax.set_ylabel("MAE")
    ax.tick_params(axis="x", rotation=45)
    plt.tight_layout()
    plt.show()
else:
    print(f"Missing {validation_summary_path}. Run validation first.")

if validation_segments_path.exists():
    validation_segments = pd.read_csv(validation_segments_path)
    final_segments = validation_segments.query("variant == @FINAL_VALIDATION_LABEL").copy()

    print("Final strategy by date")
    display(final_segments.query("segment == 'date'").sort_values("segment_value"))

    print("Final strategy by price bucket")
    display(final_segments.query("segment == 'price_bucket'").sort_values("segment_value"))

    print("Final strategy by history-count bucket")
    display(final_segments.query("segment == 'history_count_bucket'").sort_values("segment_value"))

    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    final_segments.query("segment == 'price_bucket'").sort_values("segment_value").plot(
        x="segment_value", y=["MAE", "RMSE"], kind="bar", ax=axes[0]
    )
    axes[0].set_title("Final Strategy Error by Price Bucket")
    axes[0].set_xlabel("price bucket")
    axes[0].tick_params(axis="x", rotation=0)

    final_segments.query("segment == 'date'").sort_values("segment_value").plot(
        x="segment_value", y=["MAE", "RMSE"], kind="bar", ax=axes[1]
    )
    axes[1].set_title("Final Strategy Error by Date")
    axes[1].set_xlabel("date")
    axes[1].tick_params(axis="x", rotation=0)
    plt.tight_layout()
    plt.show()
else:
    print(f"Missing {validation_segments_path}. Run validation first.")


## 10. Anchor Set vs Previous History

This section treats known prices in the test file as the anchor set and compares each anchor against historical rows from before the anchor timestamp.

The goal is to answer:

- Are anchor prices mostly equal to the last known same-`modelId` price?
- Are changes concentrated by day, shop, category, brand, or price bucket?
- Which history key is most useful for anchor correction: `modelId`, `itemId`, or `shopId`?
- Does the anchor set suggest global calibration, segment calibration, or no calibration?

For each anchor row, we compute prior same-key statistics using training rows only:

- last historical price before the anchor timestamp
- recent median over last 3 and last 7 observations
- all-history median
- count of prior observations
- log residual: `log(anchor_price) - log(history_reference_price)`


In [ ]:
anchors = test[test[config.target].notna()].copy().reset_index(drop=True)
anchors["anchor_row_id"] = np.arange(len(anchors))
anchors["anchor_log_price"] = np.log1p(anchors[config.target].clip(lower=0))

print("Anchor rows:", len(anchors))
display(
    anchors.groupby("date").agg(
        anchors=(config.target, "size"),
        unique_shopId=("shopId", "nunique"),
        unique_itemId=("itemId", "nunique"),
        unique_modelId=("modelId", "nunique"),
        median_price=(config.target, "median"),
        min_price=(config.target, "min"),
        max_price=(config.target, "max"),
    )
)


In [ ]:
def prior_stats_for_key(train_df, anchor_df, key, windows=(3, 7)):
    history_cols = [key, config.date_col, config.target]
    hist = train_df[history_cols].dropna(subset=[config.target]).copy()
    hist = hist.rename(columns={config.date_col: "hist_time", config.target: "hist_price"})
    hist[key] = hist[key].astype(str)
    hist["hist_log_price"] = np.log1p(hist["hist_price"].clip(lower=0))
    hist = hist.sort_values([key, "hist_time"])

    anchor_side = anchor_df[["anchor_row_id", key, config.date_col, config.target, "date"]].copy()
    anchor_side = anchor_side.rename(columns={config.date_col: "anchor_time", config.target: "anchor_price"})
    anchor_side[key] = anchor_side[key].astype(str)

    merged = anchor_side.merge(hist, on=key, how="left")
    merged = merged[merged["hist_time"] < merged["anchor_time"]].copy()
    merged = merged.sort_values(["anchor_row_id", "hist_time"])

    if merged.empty:
        out = anchor_df[["anchor_row_id"]].copy()
        out[f"{key}_prior_count"] = 0
        out[f"{key}_last_price"] = np.nan
        out[f"{key}_all_median_price"] = np.nan
        for w in windows:
            out[f"{key}_last_{w}_median_price"] = np.nan
        return out

    rows = []
    for anchor_id, group in merged.groupby("anchor_row_id"):
        row = {"anchor_row_id": anchor_id, f"{key}_prior_count": len(group)}
        row[f"{key}_last_price"] = group["hist_price"].iloc[-1]
        row[f"{key}_all_median_price"] = group["hist_price"].median()
        for w in windows:
            row[f"{key}_last_{w}_median_price"] = group["hist_price"].tail(w).median()
        rows.append(row)

    return pd.DataFrame(rows)

anchor_history = anchors.copy()
for key in ["modelId", "itemId", "shopId"]:
    stats = prior_stats_for_key(train, anchors, key)
    anchor_history = anchor_history.merge(stats, on="anchor_row_id", how="left")
    anchor_history[f"{key}_prior_count"] = anchor_history[f"{key}_prior_count"].fillna(0).astype(int)

for key in ["modelId", "itemId", "shopId"]:
    for ref in ["last_price", "last_3_median_price", "last_7_median_price", "all_median_price"]:
        col = f"{key}_{ref}"
        anchor_history[f"{col}_log_delta"] = (
            anchor_history["anchor_log_price"] - np.log1p(anchor_history[col].clip(lower=0))
        )
        anchor_history[f"{col}_pct_delta"] = (
            (anchor_history[config.target] - anchor_history[col]) / anchor_history[col]
        ) * 100

anchor_history.head()


### Anchor Historical Coverage

If most anchors have prior same-`modelId` history, then model-level historical priors should be very strong. If not, fallback to `itemId` or `shopId` matters more.


In [ ]:
coverage_rows = []
for key in ["modelId", "itemId", "shopId"]:
    count_col = f"{key}_prior_count"
    coverage_rows.append({
        "key": key,
        "anchors_with_history": (anchor_history[count_col] > 0).sum(),
        "anchors_without_history": (anchor_history[count_col] == 0).sum(),
        "coverage_pct": (anchor_history[count_col] > 0).mean() * 100,
        "median_prior_count": anchor_history[count_col].median(),
        "p10_prior_count": anchor_history[count_col].quantile(0.10),
        "p90_prior_count": anchor_history[count_col].quantile(0.90),
    })

pd.DataFrame(coverage_rows)


### Which Prior Best Matches Anchors?

This compares anchor price against several same-key historical references. Lower absolute log delta means the historical reference is closer to the anchor.


In [ ]:
prior_error_rows = []
for key in ["modelId", "itemId", "shopId"]:
    for ref in ["last_price", "last_3_median_price", "last_7_median_price", "all_median_price"]:
        delta_col = f"{key}_{ref}_log_delta"
        valid = anchor_history[delta_col].replace([np.inf, -np.inf], np.nan).dropna()
        prior_error_rows.append({
            "reference": f"{key}_{ref}",
            "n_valid": len(valid),
            "median_abs_log_delta": valid.abs().median(),
            "mean_abs_log_delta": valid.abs().mean(),
            "p90_abs_log_delta": valid.abs().quantile(0.90),
            "median_signed_log_delta": valid.median(),
            "pct_exact_match": (valid.abs() < 1e-9).mean() * 100 if len(valid) else np.nan,
            "pct_within_1pct_log": (valid.abs() <= 0.01).mean() * 100 if len(valid) else np.nan,
            "pct_within_5pct_log": (valid.abs() <= 0.05).mean() * 100 if len(valid) else np.nan,
        })

prior_error_summary = pd.DataFrame(prior_error_rows).sort_values("median_abs_log_delta")
prior_error_summary


In [ ]:
best_refs = prior_error_summary.head(8)["reference"].tolist()
fig, axes = plt.subplots(len(best_refs), 1, figsize=(10, max(3, 2.2 * len(best_refs))), sharex=True)
if len(best_refs) == 1:
    axes = [axes]

for ax, ref_name in zip(axes, best_refs):
    delta_col = f"{ref_name}_log_delta"
    vals = anchor_history[delta_col].replace([np.inf, -np.inf], np.nan).dropna()
    vals.clip(lower=vals.quantile(0.01), upper=vals.quantile(0.99)).hist(bins=40, ax=ax)
    ax.axvline(0, color="black", linewidth=1)
    ax.set_title(ref_name)
    ax.set_ylabel("anchors")

axes[-1].set_xlabel("anchor log price - historical reference log price")
plt.tight_layout()
plt.show()


### Anchor Shifts By Date And Segment

This shows whether anchor shifts are global or concentrated by day/shop/category/brand/price bucket. Use the best historical reference from the table above, usually same-`modelId` last price or recent median.


In [ ]:
REFERENCE_DELTA_COL = "modelId_last_price_log_delta"
REFERENCE_PCT_COL = "modelId_last_price_pct_delta"

anchor_history["anchor_price_bucket"] = pd.qcut(
    anchor_history[config.target],
    q=4,
    labels=["anchor_q1", "anchor_q2", "anchor_q3", "anchor_q4"],
    duplicates="drop",
).astype(str)

def segment_anchor_shift(df, group_col, delta_col=REFERENCE_DELTA_COL):
    valid = df.replace([np.inf, -np.inf], np.nan).dropna(subset=[delta_col])
    return (
        valid.groupby(group_col)
        .agg(
            anchors=(delta_col, "size"),
            median_log_delta=(delta_col, "median"),
            mean_abs_log_delta=(delta_col, lambda x: x.abs().mean()),
            p90_abs_log_delta=(delta_col, lambda x: x.abs().quantile(0.90)),
            exact_match_pct=(delta_col, lambda x: (x.abs() < 1e-9).mean() * 100),
            within_5pct_log_pct=(delta_col, lambda x: (x.abs() <= 0.05).mean() * 100),
        )
        .reset_index()
        .sort_values(["mean_abs_log_delta", "anchors"], ascending=[False, False])
    )

for group_col in ["date", "anchor_price_bucket", "shopId", "brand", "cat_id"]:
    print("", group_col)
    display(segment_anchor_shift(anchor_history, group_col).head(20))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

anchor_history.boxplot(column=REFERENCE_DELTA_COL, by="date", ax=axes[0])
axes[0].set_title("Anchor vs Last modelId Price: Log Delta by Date")
axes[0].set_xlabel("date")
axes[0].set_ylabel("log delta")

anchor_history.boxplot(column=REFERENCE_DELTA_COL, by="anchor_price_bucket", ax=axes[1])
axes[1].set_title("Anchor vs Last modelId Price: Log Delta by Price Bucket")
axes[1].set_xlabel("price bucket")
axes[1].set_ylabel("log delta")

plt.suptitle("")
plt.tight_layout()
plt.show()


### Implications For Anchor Utilization

Use this table to decide how anchors should affect predictions:

- If `median_signed_log_delta` is close to zero and most anchors exactly match recent history, global calibration is probably unnecessary.
- If shifts are consistent by date, use global day calibration.
- If shifts are concentrated by `shopId`, `cat_id`, or `brand`, segment calibration may help.
- If sparse-history anchors have larger shifts, second-stage or neighbor-based correction may be useful only for sparse-history rows.
- If anchor shifts are noisy, use anchors for diagnostics/model selection rather than applying corrections broadly.


## 11. Cold-Start And High-Price Diagnostics

This section focuses on the two places where the best strategy can still fail:

- **Sparse-history rows**: rows with little prior entity evidence, where fallback models matter most.
- **High-price rows**: rows where absolute errors dominate RMSE even when percentage error is low.

The first cell uses validation segments. The later cells use test anchors and engineered history features to inspect whether the shared test outage has the same risks.


In [ ]:
validation_segments_path = config.output_dir / "validation_segments.csv"

if validation_segments_path.exists():
    validation_segments = pd.read_csv(validation_segments_path)
    focus_variants = [variant for variant in REPORT_VARIANTS if variant in validation_segments["variant"].unique()]

    history_metrics = (
        validation_segments
        .query("segment == 'history_count_bucket' and variant in @focus_variants")
        .sort_values(["segment_value", "MAE"])
        .reset_index(drop=True)
    )
    price_metrics = (
        validation_segments
        .query("segment == 'price_bucket' and variant in @focus_variants")
        .sort_values(["segment_value", "MAE"])
        .reset_index(drop=True)
    )

    print("History-count segments")
    display(history_metrics)
    print("Price-bucket segments")
    display(price_metrics)
else:
    print(f"Missing {validation_segments_path}. Run validation first to populate segment diagnostics.")


In [ ]:
if validation_segments_path.exists():
    validation_segments = pd.read_csv(validation_segments_path)
    plot_variant = "hybrid_last_price_uncalibrated_fallback"

    cold_start_metrics = validation_segments.query("segment == 'history_count_bucket'").copy()
    high_price_metrics = validation_segments.query("segment == 'price_bucket'").copy()

    if plot_variant not in validation_segments["variant"].unique():
        print(f"{plot_variant} not found in validation segments.")
    else:
        fig, axes = plt.subplots(1, 2, figsize=(14, 4))

        cold_plot = cold_start_metrics.query("variant == @plot_variant").copy()
        cold_order = ["0", "1-5", "6-20", "21-100", "100+"]
        cold_plot["segment_value"] = pd.Categorical(cold_plot["segment_value"], cold_order, ordered=True)
        cold_plot = cold_plot.sort_values("segment_value")
        cold_plot.plot(x="segment_value", y=["MAE", "RMSE"], kind="bar", ax=axes[0])
        axes[0].set_title("Hybrid Error by History Count")
        axes[0].set_xlabel("fallback_entity_history_count bucket")
        axes[0].set_ylabel("Raw-price error")
        axes[0].tick_params(axis="x", rotation=0)

        price_plot = high_price_metrics.query("variant == @plot_variant").copy()
        price_order = ["price_q1", "price_q2", "price_q3", "price_q4"]
        price_plot["segment_value"] = pd.Categorical(price_plot["segment_value"], price_order, ordered=True)
        price_plot = price_plot.sort_values("segment_value")
        price_plot.plot(x="segment_value", y=["MAE", "RMSE"], kind="bar", ax=axes[1])
        axes[1].set_title("Hybrid Error by Price Bucket")
        axes[1].set_xlabel("validation price bucket")
        axes[1].set_ylabel("Raw-price error")
        axes[1].tick_params(axis="x", rotation=0)

        plt.tight_layout()
        plt.show()

        display(
            cold_start_metrics
            .query("variant == @plot_variant")
            .assign(rmse_to_mae_ratio=lambda x: x["RMSE"] / x["MAE"])
            .sort_values("RMSE", ascending=False)
        )
else:
    print(f"Missing {validation_segments_path}. Run validation first to populate segment diagnostics.")


### Test Anchor Cold-Start And High-Price Diagnostics

The hidden test rows do not have actual prices, so anchors are the only direct way to inspect current outage behavior. This uses the same historical feature builder as the model and checks whether known anchor prices are well explained by recent entity history.


In [ ]:
from src.features import build_features

# Build timestamp-safe historical features for the test outage using only train history.
test_with_history = build_features(train, test, config)
anchor_diag = test_with_history[test_with_history[config.target].notna()].copy()

anchor_diag["actual_log"] = np.log1p(anchor_diag[config.target].clip(lower=0))
anchor_diag["last_price_prior"] = np.expm1(anchor_diag["fallback_entity_price_log"]).clip(lower=0)
anchor_diag["last_price_abs_error"] = (anchor_diag[config.target] - anchor_diag["last_price_prior"]).abs()
anchor_diag["last_price_pct_error"] = np.where(
    anchor_diag[config.target] > 0,
    anchor_diag["last_price_abs_error"] / anchor_diag[config.target] * 100,
    np.nan,
)
anchor_diag["last_price_abs_log_delta"] = (
    anchor_diag["actual_log"] - anchor_diag["fallback_entity_price_log"]
).abs()
anchor_diag["history_bucket"] = pd.cut(
    anchor_diag["fallback_entity_history_count"].fillna(0),
    bins=[-0.1, 0, 5, 20, 100, np.inf],
    labels=["0", "1-5", "6-20", "21-100", "100+"],
)
anchor_diag["anchor_price_bucket"] = pd.qcut(
    anchor_diag[config.target],
    q=4,
    labels=["price_q1", "price_q2", "price_q3", "price_q4"],
    duplicates="drop",
)
anchor_diag["is_cold_start"] = anchor_diag["fallback_entity_history_count"].fillna(0).eq(0)
anchor_diag["is_high_price_anchor"] = anchor_diag["anchor_price_bucket"].astype(str).eq("price_q4")

anchor_risk_summary = pd.DataFrame({
    "anchors": [len(anchor_diag)],
    "cold_start_anchors": [int(anchor_diag["is_cold_start"].sum())],
    "cold_start_pct": [anchor_diag["is_cold_start"].mean() * 100],
    "high_price_anchors": [int(anchor_diag["is_high_price_anchor"].sum())],
    "high_price_pct": [anchor_diag["is_high_price_anchor"].mean() * 100],
    "cold_and_high_price_anchors": [int((anchor_diag["is_cold_start"] & anchor_diag["is_high_price_anchor"]).sum())],
})

display(anchor_risk_summary)

anchor_history_metrics = (
    anchor_diag
    .groupby(["history_bucket", "anchor_price_bucket"], observed=True)
    .agg(
        anchors=(config.target, "size"),
        median_price=(config.target, "median"),
        mean_price=(config.target, "mean"),
        median_history_count=("fallback_entity_history_count", "median"),
        median_abs_error=("last_price_abs_error", "median"),
        mean_abs_error=("last_price_abs_error", "mean"),
        median_pct_error=("last_price_pct_error", "median"),
        p90_pct_error=("last_price_pct_error", lambda s: s.quantile(0.90)),
        median_abs_log_delta=("last_price_abs_log_delta", "median"),
    )
    .reset_index()
    .sort_values(["history_bucket", "anchor_price_bucket"])
)

display(anchor_history_metrics)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

history_counts = anchor_diag["history_bucket"].value_counts().reindex(["0", "1-5", "6-20", "21-100", "100+"])
history_counts.plot(kind="bar", ax=axes[0], color="#4C78A8")
axes[0].set_title("Test Anchors by History Count")
axes[0].set_xlabel("history bucket")
axes[0].set_ylabel("anchor rows")
axes[0].tick_params(axis="x", rotation=0)

anchor_diag.boxplot(column="last_price_pct_error", by="anchor_price_bucket", ax=axes[1])
axes[1].set_title("Last-Price Prior Error by Anchor Price Bucket")
axes[1].set_xlabel("anchor price bucket")
axes[1].set_ylabel("absolute percentage error")
plt.suptitle("")
plt.tight_layout()
plt.show()

risk_cols = [
    config.date_col,
    "date",
    "shopId",
    "itemId",
    "modelId",
    "brand",
    "cat_id",
    config.target,
    "fallback_entity_history_count",
    "last_price_prior",
    "last_price_abs_error",
    "last_price_pct_error",
    "history_bucket",
    "anchor_price_bucket",
]

worst_anchor_priors = (
    anchor_diag[risk_cols]
    .sort_values(["last_price_abs_error", "last_price_pct_error"], ascending=False)
    .head(20)
)

display(worst_anchor_priors)

### How To Read These Diagnostics

- If `history_count_bucket = 0` or `1-5` has much worse validation error, improve only the fallback path instead of changing stable recent-entity predictions.
- If `price_q4` has much higher RMSE but similar MAPE, high absolute prices are driving RMSE rather than a broad relative-error problem.
- If test anchors with strong history have tiny last-price errors, broad day-level calibration is unnecessary and can hurt stable products.


## EDA Takeaways

- The test file is an outage-style file: most hidden rows have blank current-day metadata and price must be reconstructed.
- At `itemId` level, all shared test products appear in train. A small number of `modelId` variants are new, so item-level fallback remains important.
- Test products are generally frequent in train; the median `itemId` has many prior observations. This explains why recent entity price is the strongest signal.
- Price distributions are heavy-tailed, so the modeling pipeline uses `log1p(price)` and reports both absolute and percentage errors.
- Anchor EDA shows anchors are most useful as a guardrail for fallback rows, not as a broad correction over rows with strong recent entity history.
- The hardest residual risk is sparse or new variant history, especially for high-price products where small relative movements create large MAE/RMSE.


## Strategy Error Diagnostics

This section uses row-level validation predictions to identify where each strategy fails. 
Run `scripts/run_validation.py` first so `outputs/validation_predictions.csv` is available. 
MAPE values in this project are already percentages: `0.1424` means `0.1424%`, not `14.24%`.


In [ ]:
validation_predictions_path = config.output_dir / "validation_predictions.csv"

if validation_predictions_path.exists():
    validation_predictions = pd.read_csv(validation_predictions_path)
    validation_predictions["signed_error"] = validation_predictions["y_pred"] - validation_predictions["y_true"]
    validation_predictions["ape"] = np.where(
        validation_predictions["y_true"] != 0,
        validation_predictions["abs_error"] / validation_predictions["y_true"] * 100,
        np.nan,
    )
    validation_predictions["price_bucket"] = pd.qcut(
        validation_predictions["y_true"],
        q=4,
        labels=["price_q1", "price_q2", "price_q3", "price_q4"],
        duplicates="drop",
    ).astype(str)
    validation_predictions["history_count_bucket"] = pd.cut(
        validation_predictions["fallback_entity_history_count"].fillna(0),
        bins=[-0.1, 0, 5, 20, 100, np.inf],
        labels=["0", "1-5", "6-20", "21-100", "100+"],
    ).astype(str)

    strategy_error_summary = (
        validation_predictions
        .groupby("variant", dropna=False)
        .agg(
            rows=("y_true", "size"),
            MAE=("abs_error", "mean"),
            median_AE=("abs_error", "median"),
            p90_AE=("abs_error", lambda s: s.quantile(0.90)),
            p99_AE=("abs_error", lambda s: s.quantile(0.99)),
            MAPE=("ape", "mean"),
            bias=("signed_error", "mean"),
            overpredict_rate=("signed_error", lambda s: (s > 0).mean() * 100),
            max_AE=("abs_error", "max"),
        )
        .sort_values("MAE")
        .reset_index()
    )
    display(strategy_error_summary)
else:
    print(f"Missing {validation_predictions_path}. Run scripts/run_validation.py to create row-level diagnostics.")


In [ ]:
if validation_predictions_path.exists():
    focus_variants = [
        variant for variant in REPORT_VARIANTS
        if variant in validation_predictions["variant"].unique()
    ]
    if "hybrid_last_price_calibrated_fallback" in validation_predictions["variant"].unique():
        focus_variants.append("hybrid_last_price_calibrated_fallback")
    focus_variants = list(dict.fromkeys(focus_variants))

    segment_error_tables = {}
    for segment_col in ["date", "price_bucket", "history_count_bucket", "calibration_skip_reason"]:
        if segment_col not in validation_predictions.columns:
            continue
        table = (
            validation_predictions
            .query("variant in @focus_variants")
            .groupby(["variant", segment_col], dropna=False)
            .agg(
                rows=("y_true", "size"),
                MAE=("abs_error", "mean"),
                MAPE=("ape", "mean"),
                bias=("signed_error", "mean"),
                p90_AE=("abs_error", lambda s: s.quantile(0.90)),
            )
            .reset_index()
            .sort_values([segment_col, "MAE"], ascending=[True, True])
        )
        segment_error_tables[segment_col] = table
        print(f"Error by {segment_col}")
        display(table)


In [ ]:
if validation_predictions_path.exists():
    worst_rows = (
        validation_predictions
        .sort_values(["variant", "abs_error"], ascending=[True, False])
        .groupby("variant", group_keys=False)
        .head(10)
        .sort_values(["variant", "abs_error"], ascending=[True, False])
    )
    cols = [
        "variant", "date", "shopId", "itemId", "modelId", "cat_id", "brand",
        "y_true", "y_pred", "abs_error", "ape", "signed_error",
        "fallback_entity_history_count", "entity_weight",
        "modelId_recent_prior_count", "itemId_recent_prior_count",
        "calibration_skip_reason", "anchor_global_delta", "second_stage_delta_log",
    ]
    cols = [col for col in cols if col in worst_rows.columns]
    display(worst_rows[cols])


In [ ]:
if validation_predictions_path.exists():
    compare_variants = [
        FINAL_VALIDATION_LABEL,
        "last_price_baseline",
        "entity_blend_no_calibration",
        "global_no_calibration",
        "second_stage_residual",
    ]
    compare_variants = [v for v in compare_variants if v in validation_predictions["variant"].unique()]
    pivot_errors = validation_predictions.query("variant in @compare_variants").pivot_table(
        index=["date", "shopId", "itemId", "modelId", "y_true"],
        columns="variant",
        values="abs_error",
        aggfunc="mean",
    ).reset_index()
    error_cols = [col for col in compare_variants if col in pivot_errors.columns]
    pivot_errors["best_variant_for_row"] = pivot_errors[error_cols].idxmin(axis=1)
    best_variant_counts = (
        pivot_errors["best_variant_for_row"]
        .value_counts(normalize=False)
        .rename_axis("best_variant_for_row")
        .reset_index(name="row_count")
    )
    best_variant_counts["row_pct"] = best_variant_counts["row_count"] / len(pivot_errors) * 100
    display(best_variant_counts)


## Failure-Mode Mitigation EDA

This section investigates which proposed remedy is most promising for the remaining error tail. It asks three questions: can we identify rows where last price is unsafe, does model-level evidence matter more than item-level evidence, and which fallback strategy would have helped on those failed rows?


In [ ]:
if validation_predictions_path.exists():
    mitigation_df = validation_predictions.copy()
    final_variant = FINAL_VALIDATION_LABEL
    if final_variant not in mitigation_df["variant"].unique():
        final_variant = "hybrid_last_price_uncalibrated_fallback"

    final_errors = mitigation_df.query("variant == @final_variant").copy()
    final_errors["is_large_error"] = final_errors["ape"] > 5
    final_errors["is_tail_error"] = final_errors["abs_error"] >= final_errors["abs_error"].quantile(0.99)
    final_errors["has_model_recent"] = final_errors.get("modelId_recent_prior_count", 0).fillna(0) > 0
    final_errors["has_item_recent"] = final_errors.get("itemId_recent_prior_count", 0).fillna(0) > 0
    final_errors["evidence_type"] = np.select(
        [
            final_errors["has_model_recent"],
            final_errors["has_item_recent"],
            final_errors["fallback_entity_history_count"].fillna(0) > 0,
        ],
        ["model_recent", "item_recent_only", "historical_only"],
        default="no_entity_history",
    )

    trust_signal_summary = (
        final_errors
        .groupby(["evidence_type", "history_count_bucket", "price_bucket"], dropna=False)
        .agg(
            rows=("y_true", "size"),
            MAE=("abs_error", "mean"),
            MAPE=("ape", "mean"),
            large_error_rate=("is_large_error", "mean"),
            tail_error_rate=("is_tail_error", "mean"),
            max_AE=("abs_error", "max"),
        )
        .reset_index()
    )
    trust_signal_summary["large_error_rate"] *= 100
    trust_signal_summary["tail_error_rate"] *= 100
    trust_signal_summary = trust_signal_summary.sort_values(["large_error_rate", "MAE"], ascending=False)
    display(trust_signal_summary.head(30))
else:
    print(f"Missing {validation_predictions_path}. Run validation first.")


In [ ]:
if validation_predictions_path.exists():
    candidate_variants = [
        FINAL_VALIDATION_LABEL,
        "last_price_baseline",
        "hybrid_last_price_uncalibrated_fallback",
        "hybrid_last_price_calibrated_fallback",
        "entity_blend_no_calibration",
        "entity_blend_calibrated",
        "global_no_calibration",
        "global_calibrated",
        "second_stage_residual",
    ]
    candidate_variants = [v for v in candidate_variants if v in validation_predictions["variant"].unique()]
    error_matrix = validation_predictions.query("variant in @candidate_variants").pivot_table(
        index=["date", "shopId", "itemId", "modelId", "y_true"],
        columns="variant",
        values="abs_error",
        aggfunc="mean",
    ).reset_index()
    error_cols = [v for v in candidate_variants if v in error_matrix.columns]
    error_matrix["best_variant"] = error_matrix[error_cols].idxmin(axis=1)
    error_matrix["best_error"] = error_matrix[error_cols].min(axis=1)
    error_matrix["final_error"] = error_matrix[final_variant]
    error_matrix["avoidable_error"] = error_matrix["final_error"] - error_matrix["best_error"]

    context_cols = [
        "date", "shopId", "itemId", "modelId", "y_true",
        "evidence_type", "history_count_bucket", "price_bucket",
        "fallback_entity_history_count", "entity_weight",
        "modelId_recent_prior_count", "itemId_recent_prior_count",
        "ape", "abs_error",
    ]
    context_cols = [c for c in context_cols if c in final_errors.columns]
    error_context = final_errors[context_cols].merge(
        error_matrix[["date", "shopId", "itemId", "modelId", "y_true", "best_variant", "best_error", "final_error", "avoidable_error"]],
        on=["date", "shopId", "itemId", "modelId", "y_true"],
        how="left",
    )
    failed_rows = error_context.query("abs_error > 1").copy()

    print("Best alternative among rows where the final/last-price family made a nonzero error")
    display(
        failed_rows
        .groupby("best_variant", dropna=False)
        .agg(
            rows=("abs_error", "size"),
            mean_final_error=("final_error", "mean"),
            mean_best_error=("best_error", "mean"),
            mean_avoidable_error=("avoidable_error", "mean"),
            total_avoidable_error=("avoidable_error", "sum"),
        )
        .sort_values("total_avoidable_error", ascending=False)
        .reset_index()
    )

    print("Failure rows by evidence type and best alternative")
    display(
        failed_rows
        .groupby(["evidence_type", "best_variant"], dropna=False)
        .agg(
            rows=("abs_error", "size"),
            MAE=("abs_error", "mean"),
            mean_avoidable_error=("avoidable_error", "mean"),
        )
        .sort_values(["evidence_type", "rows"], ascending=[True, False])
        .reset_index()
    )


In [ ]:
if validation_predictions_path.exists():
    final_errors["entity_weight_bucket"] = pd.cut(
        final_errors["entity_weight"].fillna(0),
        bins=[-0.01, 0, 0.25, 0.5, 0.75, 0.9, 1.0],
        labels=["0", "0-0.25", "0.25-0.5", "0.5-0.75", "0.75-0.9", "0.9-1.0"],
    ).astype(str)
    trust_gate_candidates = []
    for weight_cutoff in [0.5, 0.75, 0.9, 0.95]:
        for require_model_recent in [False, True]:
            gate = final_errors["entity_weight"].fillna(0) >= weight_cutoff
            if require_model_recent:
                gate &= final_errors["has_model_recent"]
            gate_name = f"weight>={weight_cutoff}" + (" + model_recent" if require_model_recent else "")
            trust_gate_candidates.append({
                "gate": gate_name,
                "trusted_rows": int(gate.sum()),
                "trusted_row_pct": gate.mean() * 100,
                "trusted_MAE": final_errors.loc[gate, "abs_error"].mean(),
                "untrusted_rows": int((~gate).sum()),
                "untrusted_MAE": final_errors.loc[~gate, "abs_error"].mean(),
                "large_error_capture_pct": (final_errors.loc[~gate, "is_large_error"].sum() / max(final_errors["is_large_error"].sum(), 1)) * 100,
                "tail_error_capture_pct": (final_errors.loc[~gate, "is_tail_error"].sum() / max(final_errors["is_tail_error"].sum(), 1)) * 100,
            })
    trust_gate_candidates = pd.DataFrame(trust_gate_candidates).sort_values(
        ["large_error_capture_pct", "trusted_row_pct"], ascending=[False, False]
    )
    display(trust_gate_candidates)

    print("Error by entity-weight bucket")
    display(
        final_errors
        .groupby(["entity_weight_bucket", "evidence_type"], dropna=False)
        .agg(
            rows=("abs_error", "size"),
            MAE=("abs_error", "mean"),
            MAPE=("ape", "mean"),
            large_error_rate=("is_large_error", lambda s: s.mean() * 100),
            max_AE=("abs_error", "max"),
        )
        .reset_index()
        .sort_values(["entity_weight_bucket", "MAE"], ascending=[True, False])
    )


In [ ]:
if validation_predictions_path.exists():
    fig, axes = plt.subplots(1, 3, figsize=(18, 4))

    evidence_plot = (
        final_errors
        .groupby("evidence_type")
        .agg(MAE=("abs_error", "mean"), large_error_rate=("is_large_error", lambda s: s.mean() * 100))
        .sort_values("MAE", ascending=False)
    )
    evidence_plot["MAE"].plot(kind="bar", ax=axes[0], color="#4C78A8")
    axes[0].set_title("Final Strategy MAE by Evidence Type")
    axes[0].set_xlabel("")
    axes[0].set_ylabel("MAE")
    axes[0].tick_params(axis="x", rotation=45)

    evidence_plot["large_error_rate"].plot(kind="bar", ax=axes[1], color="#F58518")
    axes[1].set_title("Large Error Rate by Evidence Type")
    axes[1].set_xlabel("")
    axes[1].set_ylabel("Rows with APE > 5%")
    axes[1].tick_params(axis="x", rotation=45)

    if not failed_rows.empty:
        failed_rows["best_variant"].value_counts().plot(kind="bar", ax=axes[2], color="#54A24B")
    axes[2].set_title("Best Alternative on Failed Rows")
    axes[2].set_xlabel("")
    axes[2].set_ylabel("row count")
    axes[2].tick_params(axis="x", rotation=45)

    plt.tight_layout()
    plt.show()


### How To Interpret This Mitigation EDA

- If failures concentrate in `item_recent_only`, improve the modelId-vs-itemId trust gate and variant price-position features.
- If failures concentrate in low `entity_weight` or low history buckets, improve fallback calibration rather than changing strong last-price rows.
- If a global/model strategy wins many failed rows, a learned trust gate could route only risky rows away from last price.
- If no alternative strategy wins the failed rows, the misses are likely true one-off price changes that require better current-day signals or matching anchors.
